In [ ]:
# ==== THAM SỐ (cell duy nhất cần sửa) — ánh xạ 1-1 tới CLI của src.data.build_index ====
REPO_URL = "https://github.com/NguyenMinhTri24072005/MV_CAAFNet.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/COMPUTER SCIENCE/NAM4_HK1_(2026-2027)/CT208E/MV_CAAFNet"

RAW_LIDC_DIR = f"{DRIVE_ROOT}/data/raw/LIDC-IDRI"                       # --lidc-dir
RAW_LUNA16_DIR = f"{DRIVE_ROOT}/data/raw/LUNA16"                        # --luna16-dir
INDEX_OUT_DIR = f"{DRIVE_ROOT}/data/index"                              # --out-dir
WORKERS = 4                                                             # --workers
RUN_BUILD_INDEX = True                                                  # False nếu data/index đã có sẵn trên Drive
DATA_CFG = "configs/data.yaml"
SAMPLE_HU_SCANS = 50                                                    # số scan lấy mẫu vẽ histogram HU (B9)
SAMPLE_OVERLAY_SCANS = 5                                                # số scan LUNA16 vẽ chồng mask kiểm tra bằng mắt

# 00 — Khảo sát dữ liệu & lập chỉ mục (Phase 1)

**Runtime:** Colab Pro **CPU High-RAM** (Runtime → Change runtime type → CPU, High-RAM). Không cần GPU.

- Chỉ sửa **cell tham số** (cell đầu tiên, ngay trên); mọi cell khác đọc từ đó.
- Repo private: thêm secret `GH_TOKEN` trong Colab Secrets (biểu tượng 🔑) và bật quyền truy cập cho notebook.
- Kết quả ghi lên Drive: `DRIVE_ROOT/data/index/` (`ct_scans.csv`, `seg_index.csv`, `luna16_scans.csv`, `index_summary.json`,
  và các tệp khảo sát `survey_*`).
- Cuối notebook là **cổng quyết định**: chốt tham số vào `configs/data.yaml` rồi commit từ máy local.

In [ ]:
# Setup chung: mount Drive → clone/pull repo (GH_TOKEN qua GIT_ASKPASS, không in token) → cài thư viện
import os, platform, shlex, subprocess, sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/MV_CAAFNet"


def _git_env():
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")          # Colab Secrets → không ghi token vào notebook
    except Exception:
        token = None                               # repo public hoặc chưa cấu hình secret
    if token:
        askpass = Path("/content/.git_askpass.sh")
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  Username*) echo "x-access-token" ;;\n  *) printf "%s" "$GH_TOKEN" ;;\nesac\n')
        askpass.chmod(0o700)
        env.update(GIT_ASKPASS=str(askpass), GH_TOKEN=token)
    return env


def _git(*args):
    subprocess.run(["git", *args], check=True, env=_git_env())


if Path(REPO_DIR, ".git").is_dir():
    _git("-C", REPO_DIR, "fetch", "--quiet", "origin", BRANCH)
    _git("-C", REPO_DIR, "checkout", "--quiet", BRANCH)
    _git("-C", REPO_DIR, "pull", "--quiet", "--ff-only", "origin", BRANCH)
else:
    _git("clone", "--quiet", "--branch", BRANCH, REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)


def run(args):
    """Chạy `python <args>` từ thư mục repo (danh sách tham số → an toàn với đường dẫn có dấu cách)."""
    cmd = [sys.executable, *map(str, args)]
    print("$", " ".join(shlex.quote(c) for c in cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=REPO_DIR)


commit = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
mem_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024 ** 3
print(f"commit {commit} | Python {platform.python_version()} | {os.cpu_count()} vCPU | RAM {mem_gb:.1f} GB")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available())
except ImportError:
    print("PyTorch: không có (không cần cho notebook CPU này)")

## 1. Lập chỉ mục (chỉ đọc header DICOM và tệp text `.mhd`)

In [ ]:
if RUN_BUILD_INDEX:
    run(["-m", "src.data.build_index", "--lidc-dir", RAW_LIDC_DIR, "--luna16-dir", RAW_LUNA16_DIR,
         "--out-dir", INDEX_OUT_DIR, "--workers", WORKERS])
else:
    print("Bỏ qua build_index, dùng chỉ mục có sẵn tại", INDEX_OUT_DIR)

## 2. Toàn vẹn cấu trúc (P1.1–1.2)

In [ ]:
import json

import numpy as np
import pandas as pd
from IPython.display import display

from src.utils import load_config

data_cfg = load_config(DATA_CFG)
idx = Path(INDEX_OUT_DIR)
summary = json.loads((idx / "index_summary.json").read_text(encoding="utf-8"))
ct = pd.read_csv(idx / "ct_scans.csv", dtype={"series_uid": str, "patient_id": str})
seg = pd.read_csv(idx / "seg_index.csv", dtype={"ref_series_uid": str, "patient_id": str, "nodule_key": str, "seg_rel_path": str})
luna = pd.read_csv(idx / "luna16_scans.csv", dtype={"series_uid": str, "patient_id": str, "mhd_rel_path": str})
print(json.dumps({k: summary[k] for k in ("lidc", "luna16", "n_errors")}, ensure_ascii=False, indent=2))
if summary["n_errors"]:
    display(pd.DataFrame(summary["errors"]).head(50))


def check(ok, msg):
    print(("✔ " if ok else "✘ ") + msg)


anom = seg[seg["label_anomaly"].astype(bool)]
check(len(anom) == 0, f"mọi SEG đều parse được 'Nodule <K>' ({len(anom)} tệp dị biệt — dùng nodule_key dự phòng)")
if len(anom):
    display(anom[["seg_rel_path", "segment_label", "series_desc", "nodule_key"]].head(50))
check(summary["lidc"]["n_seg_ref_not_in_patient_ct"] == 0, "mọi SEG trỏ tới một CT của chính bệnh nhân đó")
pos = luna[luna["n_annotations"] > 0]
n_pos_seg = int((pos["n_seg_files"] > 0).sum())
check(n_pos_seg == len(pos), f"LUNA16 dương tính có SEG: {n_pos_seg}/{len(pos)} (kỳ vọng 601/601)")
check(len(luna) == 888 and summary["luna16"]["n_annotations"] == 1186, f"LUNA16: {len(luna)} scan, {summary['luna16']['n_annotations']} nốt (kỳ vọng 888 / 1.186)")
check(summary["lidc"]["n_ct_scans"] == 1018, f"LIDC: {summary['lidc']['n_ct_scans']} CT scan (kỳ vọng 1.018)")

## 3. Phân bố LIDC-IDRI (P1.3–1.4)

Đếm số lát có mask đồng thuận theo `min_readers ∈ {1..4}` × `min_mask_area_px ∈ {1, 5, 10}` (chỉ đọc pixel SEG,
không đọc CT). Bảng **chưa áp lọc đường kính ≥ 3 mm**, nên là cận trên của số patch thực tế.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

from tqdm.auto import tqdm

from src.data.preprocess_lidc import cluster_slice_areas

clevel = float(data_cfg["lidc"]["clevel"])
clusters = (seg.groupby(["patient_id", "ref_series_uid", "nodule_key"])["seg_rel_path"]
               .apply(list).reset_index())
clusters["n_readers"] = clusters["seg_rel_path"].map(len)
cache = idx / f"survey_cluster_areas_clevel{clevel}.json"
if cache.exists():
    areas = json.loads(cache.read_text(encoding="utf-8"))
else:
    with ThreadPoolExecutor(WORKERS) as pool:
        areas = list(tqdm(pool.map(lambda p: cluster_slice_areas(RAW_LIDC_DIR, p, clevel), clusters["seg_rel_path"]),
                          total=len(clusters), desc="SEG"))
    cache.write_text(json.dumps(areas), encoding="utf-8")
clusters["areas"] = areas

print(f"{ct['patient_id'].nunique()} bệnh nhân, {len(ct)} scan, {len(clusters)} cụm nốt")
display(clusters["n_readers"].value_counts().sort_index().rename("số cụm").to_frame())
TARGET = 13928 + 1200
table = pd.DataFrame(
    {a: [sum(sum(x >= a for x in ar) for ar, n in zip(clusters["areas"], clusters["n_readers"]) if n >= r)
         for r in (1, 2, 3, 4)] for a in (1, 5, 10)},
    index=pd.Index([1, 2, 3, 4], name="min_readers"),
)
table.columns.name = "min_mask_area_px"
display(table)
best = (table - TARGET).abs().stack().idxmin()
print(f"Gần {TARGET} nhất: min_readers={best[0]}, min_mask_area_px={best[1]} → {table.loc[best]} lát")

## 4. Khớp 888 scan / 1.186 nốt LUNA16 ↔ DICOM-SEG (P1.5 — B15)

Chỉ đọc **header** `.mhd` (không đọc `.raw`) và pixel SEG, rồi chạy đúng bước ánh xạ tọa độ + khớp 1-1 của §5.3.

In [ ]:
from src.data.build_index import load_annotations
from src.data.preprocess_luna16 import MhdGeometry, build_luna_clusters, match_clusters

luna_clevel = float(data_cfg["luna16"]["clevel"])
ann = load_annotations(Path(RAW_LUNA16_DIR), data_cfg["luna16"]["annotations_csv"])
ann_by = {u: g[["coordX", "coordY", "coordZ", "diameter_mm"]].astype(float).to_dict("records")
          for u, g in ann.groupby("seriesuid", sort=False)}
seg_by = {u: g[["seg_rel_path", "nodule_key", "nodule_num", "label_anomaly"]].to_dict("records")
          for u, g in seg.groupby("ref_series_uid")}


def check_scan(row):
    try:
        geom = MhdGeometry.from_file(Path(RAW_LUNA16_DIR) / row.mhd_rel_path)
        sx, sy, sz = geom.GetSize()
        cl = build_luna_clusters(geom, seg_by.get(row.series_uid, []), RAW_LIDC_DIR, luna_clevel, (sz, sy, sx))
        _, rep = match_clusters(cl, geom, ann_by[row.series_uid], row.series_uid)
        n_general = sum(c["n_general_frames"] for c in cl)
        return [dict(r, n_clusters=len(cl), n_general_frames=n_general, error="") for r in rep]
    except Exception as exc:  # ghi lại, không dừng cả lượt khảo sát
        return [{"series_uid": row.series_uid, "status": "error", "error": f"{type(exc).__name__}: {exc}"}]


with ThreadPoolExecutor(WORKERS) as pool:
    reports = list(tqdm(pool.map(check_scan, pos.itertuples(index=False)), total=len(pos), desc="LUNA16"))
match_df = pd.DataFrame([r for rep in reports for r in rep])
match_df.to_csv(idx / "survey_match_report.csv", index=False)
counts = match_df["status"].value_counts()
display(counts.rename("số nốt").to_frame())
print(f"Khớp 1-1: {counts.get('matched', 0)}/{len(ann)} nốt ({100 * counts.get('matched', 0) / max(len(ann), 1):.2f}%)")
print("Scan có Direction ≠ I:", int((~luna['direction_is_identity'].astype(bool)).sum()),
      "| SEG có IOP ≠ đơn vị:", summary["lidc"]["n_seg_iop_not_identity"])
display(match_df[match_df["status"] != "matched"].head(50))

### Vẽ chồng mask đồng thuận lên lát `.mhd` (ưu tiên scan có Direction ≠ I)

In [ ]:
import matplotlib.pyplot as plt
import SimpleITK as sitk
from scipy import ndimage

from src.data.ct_utils import window_hu
from src.data.preprocess_luna16 import world_to_voxel_batch

cand = pos[pos["n_seg_files"] > 0].sort_values(["direction_is_identity", "series_uid"])
pick = list(cand.head(SAMPLE_OVERLAY_SCANS).itertuples(index=False))
fig, axes = plt.subplots(1, len(pick), figsize=(4.2 * len(pick), 4.4), squeeze=False)
for ax, row in zip(axes[0], pick):
    img = sitk.ReadImage(str(Path(RAW_LUNA16_DIR) / row.mhd_rel_path))
    vol = sitk.GetArrayFromImage(img)
    cl = build_luna_clusters(img, seg_by[row.series_uid], RAW_LIDC_DIR, luna_clevel, vol.shape)
    assign, _ = match_clusters(cl, img, ann_by[row.series_uid], row.series_uid)
    cm = cl[next(iter(assign.values()))[0]]["cmask"] if assign else cl[0]["cmask"]
    cz, cy, cx = (int(round(v)) for v in ndimage.center_of_mass(cm))
    y0, x0 = max(cy - 48, 0), max(cx - 48, 0)
    ax.imshow(window_hu(vol[cz, y0:cy + 48, x0:cx + 48]), cmap="gray")
    ax.contour(cm[cz, y0:cy + 48, x0:cx + 48], levels=[0.5], colors="r", linewidths=1)
    for a in ann_by[row.series_uid]:
        ix, iy, iz = world_to_voxel_batch(img, np.array([[a["coordX"], a["coordY"], a["coordZ"]]]))[0]
        if abs(iz - cz) < 2:
            ax.plot(ix - x0, iy - y0, "c+", ms=12)
    ax.set_title(f"…{row.series_uid[-8:]} z={cz} Dir=I:{row.direction_is_identity}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Chốt giả định B6, B8, B9, B10, B17 (P1.6–1.9)

Lấy mẫu `SAMPLE_HU_SCANS` scan LUNA16 dương tính: đọc `.mhd`, dựng mask đồng thuận đã khớp, cắt patch đúng như §5.3.

In [ ]:
from src.data.ct_utils import compute_nodule_diameters
from src.data.preprocess_lidc import extract_scan_patches

sample = cand.sample(min(SAMPLE_HU_SCANS, len(cand)), random_state=0)
hu_vals, patch_rows = [], []
rng = np.random.default_rng(0)
for row in tqdm(list(sample.itertuples(index=False)), desc="mẫu HU"):
    img = sitk.ReadImage(str(Path(RAW_LUNA16_DIR) / row.mhd_rel_path))
    vol = sitk.GetArrayFromImage(img).astype(np.float32)
    sx, sy, sz = img.GetSpacing()
    cl = build_luna_clusters(img, seg_by[row.series_uid], RAW_LIDC_DIR, luna_clevel, vol.shape)
    assign, _ = match_clusters(cl, img, ann_by[row.series_uid], row.series_uid)
    nodules = []
    for a, (k, _) in assign.items():
        vals = vol[cl[k]["cmask"] > 0]
        hu_vals.append(rng.choice(vals, size=min(len(vals), 5000), replace=False))
        _, d2 = compute_nodule_diameters(cl[k]["cmask"], (sz, sy, sx))
        nodules.append({"nodule_idx": cl[k]["nodule_idx"], "n_readers": cl[k]["n_readers"],
                        "diameter_mm": ann_by[row.series_uid][a]["diameter_mm"], "diameter_2d_mm": d2,
                        "cmask": cl[k]["cmask"]})
    if nodules:
        patch_rows += extract_scan_patches(vol, (sz, sy, sx), nodules, patient_id=row.patient_id,
                                           series_uid=row.series_uid, scan_idx=row.scan_idx,
                                           common=data_cfg["common"])[2]
hu = np.concatenate(hu_vals)
pct = dict(zip([1, 5, 50, 95, 99], np.percentile(hu, [1, 5, 50, 95, 99]).round(1)))
pr = pd.DataFrame(patch_rows)
print("B9  HU trong mask (percentile):", pct, "→ gợi ý hu_window:", "[-1000, 150]" if pct[95] <= 150 else "[-1000, 400]")
print("B6  area_nodule_px:", pr["area_nodule_px"].describe(percentiles=[.05, .1, .25, .5]).round(1).to_dict())
print(f"    lát < 5 px: {(pr['area_nodule_px'] < 5).mean():.1%}, < 10 px: {(pr['area_nodule_px'] < 10).mean():.1%}")
print(f"B8  patch có nốt khác lọt vào (n_other_nodules > 0): {(pr['n_other_nodules'] > 0).mean():.2%} (ngưỡng quan tâm 2%)")
print(f"B17 patch chạm biên (touches_border): {pr['touches_border'].mean():.2%}")
spx = ct["pixel_spacing_x"]
print(f"B10 PixelSpacing LIDC: median {spx.median():.3f} mm, < 1 mm: {(spx < 1).mean():.1%} → "
      f"{'giữ resample_mm: null' if (spx < 1).mean() > 0.5 else 'cân nhắc resample_mm: 1.0'}")
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(hu, bins=120, range=(-1100, 600)); ax[0].set_title("HU trong mask đồng thuận (B9)")
for v in (150, 400):
    ax[0].axvline(v, color="r", ls="--")
ax[1].hist(pr["area_nodule_px"], bins=60); ax[1].set_title("area_nodule_px (B6)")
plt.tight_layout()
plt.show()

## 6. Cổng quyết định Phase 1

1. Ghi các giá trị đã chốt (`lidc.min_readers`, `common.min_mask_area_px`, `common.hu_window`, `common.resample_mm`,
   `common.include_other_nodules`) vào `configs/data.yaml` **ở máy local**, commit và push.
2. Nếu số patch ước lượng lệch > 20% so với Table 4 (13.928 + 1.200), ghi nguyên nhân vào `README.md` (không ép số).
3. Sang `01_colab_preprocess.ipynb`.